# Skill loading, studied

How Claude Code loads a skill, reproduced here with `bro_skills/`:

1. **Discovery** — scan every `bro_skills/*/SKILL.md`, parse only the YAML frontmatter (`name`, `description`). This is cheap and happens for *all* skills up front — like the skill listing Claude sees in its system prompt.
2. **Selection** — pick a skill by matching intent against the descriptions (here: just by name, to keep it simple).
3. **Load** — read the *full* SKILL.md body only for the chosen skill. This is the "progressive disclosure" part: full instructions are loaded lazily, not for every skill.
4. **Use** — feed the loaded instructions to an LLM call (or in this notebook, just print them, since no LLM call is wired up yet).

The plumbing this notebook exercises now lives in `src/bro_agent/` (see the
package's own docstrings for detail) so the sales-funnel study in
`sales_agent.ipynb` can reuse it without copy-pasting cells. Each cell here
imports one small piece and demonstrates it.

In [ ]:
from bro_agent.skills import SKILLS_DIR, discover_skills, load_skill, skill_dir

skill_registry = discover_skills()
skill_registry

## Stage 2 — load a chosen skill's full instructions

`skill_registry` only has name + description. Loading the body happens
only for the one skill actually picked -- that's the lazy part.

In [ ]:
instructions = load_skill("hello-world", skill_registry)
print(instructions)

## Interactive skills — tools the kernel can actually run

A skill is instructions, not code. `grep-file/SKILL.md` tells the agent to
call `grep_file(pattern, path)` — here are those tools, and re-running
discovery now that a second skill exists.

In [ ]:
from bro_agent.execution import TOOLS, read_file, grep_file

# re-discover now that bro_skills/ has a second skill
skill_registry = discover_skills()
skill_registry

## Routing — letting a model pick the skill

This is the part Claude Code does with a real LLM call: hand the model
every `(name, description)` pair plus the user's request, and let it
choose. Wired here through `brollm.BaseContract` to Bedrock's `converse`
API, model `google.gemma-3-4b-it` in `us-east-1`.

In [ ]:
from bro_agent.llm import router, build_router_prompt, call_bedrock

chosen = router(user_request="find every place we log errors in dev.ipynb", registry=skill_registry)
print(chosen)
instructions = load_skill(chosen, skill_registry)
print(instructions)

In [5]:
r_prompt = build_router_prompt("find every place we log errors in dev.ipynb", skill_registry)
print(r_prompt)

Given this user request, pick the single best-matching skill by name.
Reply with only the skill name, nothing else.

Skills:
- grep-file: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.
- hello-world: A minimal example skill. Greets a person by name and explains what it just did. Use this to learn how skill discovery and loading works.
- word-count: Count words per line in a text file using the bundled count_words.py script. Use this when the user asks how many words are on each line, or wants a per-line word count, of a specific file.

User request: find every place we log errors in dev.ipynb


## Executing the chosen skill

Routing only produces a name. Executing means: ask the model, given the
skill's instructions, to decide which tool to call and with what
arguments; run that tool for real in the kernel; then feed the result
back so the model can write the final answer following the skill's
instructions. Three steps, so this is a `broflow` `Flow` — each step
decides its own next step, exactly like `PlanTask` here choosing `act`
when a tool is needed and `respond` when it isn't (e.g. `hello-world`
needs no tool at all).

In [ ]:
from bro_agent.execution import run_script, execute_skill, skill_flow, PlanTask, ActTask, RespondTask

# The three-step plan/act/respond flow itself (PlanTask/ActTask/RespondTask,
# wired into a broflow.Flow) now lives in src/bro_agent/execution.py --
# import it instead of redefining it here. See that file's docstrings for
# how each step decides its own next step.

In [ ]:
user_request = "grep for the word 'description' in bro_skills/grep-file/SKILL.md"

chosen = router(user_request=user_request, registry=skill_registry)
print("routed to:", chosen)

result = execute_skill(chosen, user_request, skill_registry)
print(skill_flow.trace)
print(result["answer"])

## Bundled scripts — a skill can carry its own executable

`word-count/` bundles `count_words.py` next to its `SKILL.md`. `run_script`
runs it as a real subprocess (`sys.executable script.py arg1 arg2 ...`,
never `shell=True`) inside the skill's own folder, and only stdout/stderr
comes back — the script's source is never read into the model's context,
unlike `TOOLS` functions which run in-process. `build_plan_prompt` lists
whatever `*.py` files sit in a skill's folder so the model knows it can
call `run_script` at all.

In [ ]:
# re-discover so the new word-count skill folder is picked up
skill_registry = discover_skills()

user_request = "count words per line in bro_skills/word-count/sample.txt"
chosen = router(user_request=user_request, registry=skill_registry)
print("routed to:", chosen)

result = execute_skill(chosen, user_request, skill_registry)
print(skill_flow.trace)
print(result["tool_result"])
print(result["answer"])